# Використання агентів разом з RAG

## Звичайний конвеєр

Створимо звичайний конвеєр RAG.

In [1]:
from dotenv import load_dotenv
from google import genai

load_dotenv()
gemini_client = genai.Client()

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [3]:
from rag_helper import RAGBase

instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(index, gemini_client, instructions)

Ставимо питання.

In [4]:
answer = assistant.rag('How do I run Ollama locally?')
print(answer)

/workspaces/LLM_Zoomcamp_2026/rag_helper.py:77: UserWarning: Interactions usage is experimental and may change in future versions.
  interaction = self.llm_client.interactions.create(


To run Ollama locally, follow these steps:

1.  **Install Ollama:** Visit [https://ollama.com/download](https://ollama.com/download) and download the installer for your operating system (macOS, Windows, or Linux). For Linux, you can run the command: `curl -fsSL https://ollama.com/install.sh | sh`.
2.  **Start the model:** Once installed, open a terminal and run `ollama run llama3`. This will download the model, start it locally, and open a chat interface.
3.  **Test the server:** You can verify the local server is running by executing `curl http://localhost:11434`.
4.  **Use Python:** To interact with Ollama via Python, install the client with `pip install ollama`. You can then use the `ollama.chat` function in your code to send prompts to the model.


А тепер допускаємо помилку в слові Ollama (пропускаємо букву "l").

In [5]:
answer = assistant.rag('How do I run Olama locally?')
print(answer)

The provided FAQ database does not contain information on how to run Ollama locally.


## Додаємо агента

Спочатку ми визначаємо функцію пошуку для моделі. Вона така сама, як в модулі rag_helper.py. Але викликати її буде вже безпосередньо сама модель.

In [ ]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

Далі ми розповідаємо моделі про цю функцію. Модель не бачить нашого коду Python, лише схему, яка описує, що робить функція та які аргументи вона приймає. LLM не залежать від мови. Зрештою, ми просто здійснюємо HTTP-виклик, тому ми описуємо інструмент у JSON, а не в Python. Та сама схема працюватиме з TypeScript або Java.

In [7]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

Description - найважливіше поле, оскільки модель зчитує його, щоб вирішити, коли викликати функцію. Parameters - це схема JSON для аргументів, і ми позначаємо поле query як обов'язкове, щоб модель завжди заповнювала його.

In [8]:
message = "I just discovered the course. Can I join it?"

interaction = gemini_client .interactions.create(
    model='gemini-3.1-flash-lite',
    input=message,
    tools=[search_tool]
)

interaction.output_text

''